This notebook collects heat hazard metrics for all census tracts in California, using the following indicators:

    - Heat days over 90F, historic and projected (mid-century, 2040-2070)
    - Heat days over 90F delta, projected - historic
    - 90F days (historic) are then normalized, 0-100
    - Air quality: combined normalized Ozone and PM2.5, historic

Heat and air quality are then multipled to create
    - heat_hazard_idx_norm per census tract

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd

## Import Data

In [14]:
ces = gpd.read_file("data_sources/hazards/CalEnviroScreen_4.0_Results.geojson")
len(ces)

8035

In [15]:
ces['tract'] = ces['tract'].astype(str)
ces['tract'] = ces['tract'].str.zfill(11)

In [16]:
ces_cols = ['tract', 'PollutionP', 'ozoneP', 'pmP', 'dieselP']
ces = ces[ces_cols]

In [17]:
fca = pd.read_csv("data_sources/hazards/heatdays_alltimes_tract.csv", dtype={"GEOID":object})
len(fca)

9109

In [18]:
fca_cols = [
    'GEOID',
    'days_over_80_historic',
    'days_over_80_midcentury',
    'days_over_90_historic',
    'days_over_90_midcentury',
    'days_over_100_historic',
    'days_over_100_midcentury',
]
fca = fca[fca_cols]

In [19]:
temp_thresholds = ["80", "90", "100"]
for temp in temp_thresholds:
    fca[f'delta_{temp}'] = fca[f'days_over_{temp}_midcentury'] - fca[f'days_over_{temp}_historic']

In [20]:
indices_data = pd.merge(fca, ces, left_on='GEOID', right_on='tract', how='left')
indices_data = indices_data.drop(columns=["tract"])

## Normalize

In [24]:
def normalize_indicators(df, columns):
    """
    Normalizes columns to a 0-1 scale using Min-Max scaling.
    Used for days/counts that aren't already percentiles.
    """
    for col in columns:
        col_min = df[col].min()
        col_max = df[col].max()
        if col_max != col_min:
            df[f'{col}_norm'] = ((df[col] - col_min) / (col_max - col_min)) * 100
        else:
            df[f'{col}_norm'] = 0
    return df

In [25]:
heat_cols = [
    'days_over_80_historic', 'days_over_80_midcentury',
    'days_over_90_historic', 'days_over_90_midcentury',
    'days_over_100_historic', 'days_over_100_midcentury',
]
indices_norm = normalize_indicators(indices_data, heat_cols)

## Create Indicies

In [32]:
indices_norm['AQI'] = indices_norm[['ozoneP', 'pmP', 'dieselP']].mean(axis=1)

In [33]:
indices_norm = normalize_indicators(indices_norm, ['AQI'])

In [39]:
indices_norm['heat_hazard_idx'] = indices_norm[['AQI_norm', 'days_over_90_historic_norm']].mean(axis=1)

In [40]:
idx = normalize_indicators(indices_norm, ['heat_hazard_idx'])

## Export, keeping this limited for now

In [43]:
cols_to_keep = [
    'days_over_90_historic',
    'days_over_90_midcentury',
    'delta_90',
    'PollutionP',
    'AQI_norm',
    'heat_hazard_idx_norm'
]
idx = idx[cols_to_keep]

In [46]:
idx

,days_over_90_historic,days_over_90_midcentury,delta_90,PollutionP,AQI_norm,heat_hazard_idx_norm
0,85.056664,114.804840,29.748177,24.293715,43.485370,43.810232
1,77.778118,108.555836,30.777718,55.743622,58.317556,49.357509
2,77.037288,108.080700,31.043413,84.331052,42.263339,41.121663
3,85.446671,115.751076,30.304405,62.924704,45.258176,44.799480
4,79.092915,110.039068,30.946152,60.584941,53.498279,47.283197
...,...,...,...,...,...,...
9104,53.225180,91.522087,38.296907,6.397013,37.112737,32.375831
9105,7.246115,15.753077,8.506962,68.064717,51.962134,27.911934
9106,78.203186,113.866723,35.663537,18.531425,48.300344,44.448391
9107,91.289132,127.747498,36.458365,77.112632,87.934596,67.695349


In [44]:
idx.to_csv("data/heat_air_hazard.csv")